### Importing libs


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Loading the Data

In [ ]:
df = pd.read_csv('healthcare_dataset.csv')

In [ ]:
df.shape

### Inspection of Data


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

### Cleaning Data

#### 1.Removing duplicates

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

#### 2.Standardizing Names

In [ ]:
df['Name']=df['Name'].str.strip().str.title()

In [ ]:
df.head()

#### 3.Fixing Billing Amounts

In [ ]:
df.loc[df['Billing Amount']<0,'Billing Amount']=df['Billing Amount'].mean()

In [ ]:
df.describe()

In [ ]:
df.groupby('Medical Condition')['Billing Amount'].mean().sort_values(ascending = False)

In [ ]:
df[["Date of Admission","Discharge Date"]] = df[["Date of Admission","Discharge Date"]].apply(pd.to_datetime)

In [ ]:
df.info()

In [ ]:
df['Length_of_stay'] = (df['Discharge Date']-df['Date of Admission']).dt.days

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

### Data Analysis and Data Visualisation

#### 1.which condition affects the patient most across the network 

In [ ]:
condition_counts = df['Medical Condition'].value_counts()
condition_counts

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(condition_counts.index,condition_counts.values,color='blue')
plt.title('Patients by medical condition')
plt.xlabel('Medical condition')
plt.ylabel('Patient counts')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

#### How is patient age distributed across the network

In [ ]:
avg_age = df['Age'].mean()
avg_age

In [ ]:
plt.figure(figsize = (8,5))
plt.hist(df['Age'],bins=20,color = 'red',edgecolor = 'black')
plt.title('Age distribution')
plt.xlabel('Age')
plt.ylabel('Number of patients')
plt.tight_layout()
plt.show()


In [ ]:
df[df["Age"].between(13, 18)].shape[0]

####  how has avg treatment cost changed over time

In [ ]:
df['admission_month'] = df['Date of Admission'].dt.to_period('M')
monthly_avg_cost = df.groupby('admission_month')['Billing Amount'].mean()
monthly_avg_cost



In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly_avg_cost.index.astype(str),monthly_avg_cost.values,color = 'green',marker ='o',markersize=3)
plt.title('Average Treatment Cost Over Time (Monthly)')
plt.xlabel('Months')
plt.ylabel('Avg Billing Amount')
plt.xticks(rotation = 90)
plt.tight_layout()
plt.show()

 #### Is there any relation between patient age and treatment cost

In [ ]:
correlation = df["Age"].corr(df["Billing Amount"])
print("Correlation:", round(correlation, 3))

In [ ]:
sample_df = df.sample(1000,random_state =1)
plt.figure(figsize=(8,5))
plt.scatter(sample_df['Age'],sample_df['Billing Amount'],color='purple',alpha = 0.5,s=15)
plt.title('Age vs Treatment Cost')
plt.xlabel('Age')
plt.ylabel('Treatment Cost')
plt.tight_layout()
plt.show()


####  Which insurance provider covers the most  patient

In [ ]:
insurance_count = df['Insurance Provider'].value_counts()
insurance_count

In [ ]:
plt.figure(figsize=(8,5))
plt.pie(insurance_count.values,labels=insurance_count.index,autopct='%1.1f%%',colors=['red','blue','green','purple','yellow'])
plt.title('Patient Share By Insurance')
plt.tight_layout()
plt.show()

## AD-HOC  Business Questions

#### 1.Emergency Department Capacity Planning


Business scenario :- "The hostpital management is planning staffing for emergency cases. They want to know how many emergncy cases occured in each month" 

In [ ]:
emergency_month = (df.loc[df['Admission Type']== 'Emergency'].groupby(df['Date of Admission'].dt.month).size())
emergency_month

### 2.Identify High Cost Patients Groups

Businesss scenario:- "Hospital finance team wants to identify patients who have unusually high billing amount so that they can review those cases"

In [ ]:
high_cost_patients = df.loc[df['Billing Amount']>df['Billing Amount'].quantile(0.90)]
high_cost_patients


### 3.Flagging emergency cases with abnormal results for review

Business scenario :- "Hospital management wants to identify emergency patients whos test results are abnormal,so these cases can be prioritize for review"

In [ ]:
Emergency_patients = df.loc[(df['Admission Type']=='Emergency') & (df['Test Results']=='Abnormal')]
Emergency_patients.shape[0]

In [ ]:
Emergency_patients